# 🧹 Limpieza de Datos — Airbnb Listings Málaga

Notebook de limpieza preparado para ser trasladado a un DAG de Airflow.  
Cada sección corresponde a una **tarea (task)** independiente del pipeline.  
Tras cada task se guarda una copia del dataframe en el diccionario `checkpoints`.

---

| Task ID (DAG)            | Checkpoint guardado     | Descripción                                    |
|:-------------------------|:------------------------|:-----------------------------------------------|
| `load_raw_data`          | `01_loaded`             | Carga del CSV crudo                            |
| `drop_high_null_cols`    | `02_drop_nulls`         | Eliminar columnas con >80% nulos               |
| `clean_price`            | `03_price`              | Limpiar y convertir `price` a float            |
| `clean_booleans`         | `04_booleans`           | Homogeneizar columnas booleanas (`t`/`f`)      |
| `clean_rates`            | `05_rates`              | Limpiar tasas de respuesta/aceptación          |
| `clean_dates`            | `06_dates`              | Parsear columnas de fecha                      |
| `clean_bathrooms`        | `07_bathrooms`          | Validar columna `bathrooms` numérica           |
| `engineer_features`      | `08_features`           | Crear variables derivadas                      |
| `flag_outliers`          | `09_outliers`           | Marcar outliers (sin eliminar filas)           |


---
## 0. Imports y configuración

In [ ]:
import pandas as pd
import numpy as np
import re

RAW_PATH    = '../data/listings.csv'
SCRAPE_DATE = pd.Timestamp('2025-09-30')

# Diccionario donde se acumulan las copias del df tras cada task
checkpoints = {}



---
## Task 1 — `load_raw_data`
Carga del CSV crudo y comprobación mínima de integridad.

In [27]:
df = pd.read_csv(RAW_PATH, low_memory=False)

print(f'Filas: {df.shape[0]:,}   Columnas: {df.shape[1]}')

REQUIRED_COLS = [
    'id', 'listing_url', 'name', 'price', 'room_type',
    'accommodates', 'bathrooms', 'bathrooms_text',
    'bedrooms', 'beds', 'latitude', 'longitude',
    'host_since', 'host_is_superhost', 'neighbourhood_cleansed',
    'amenities', 'minimum_nights', 'availability_365',
    'number_of_reviews', 'review_scores_rating'
]

missing_cols = [c for c in REQUIRED_COLS if c not in df.columns]
assert len(missing_cols) == 0, f' Columnas faltantes: {missing_cols}'
assert df['id'].nunique() == len(df), ' IDs duplicados detectados'

checkpoints['01_loaded'] = df.copy()
print(f'checkpoint["01_loaded"] guardado — {checkpoints["01_loaded"].shape}')
print('load_raw_data OK ')


Filas: 9,714   Columnas: 79
checkpoint["01_loaded"] guardado — (9714, 79)
load_raw_data OK 


---
## Task 2 — `drop_high_null_cols`
Eliminar columnas con más del 80% de valores nulos.

In [7]:
df = checkpoints['01_loaded'].copy()

NULL_THRESHOLD = 0.80
null_pct  = df.isnull().mean()
drop_cols = null_pct[null_pct > NULL_THRESHOLD].index.tolist()

print(f'Columnas eliminadas ({len(drop_cols)}): {drop_cols}')
df.drop(columns=drop_cols, inplace=True)

checkpoints['02_drop_nulls'] = df.copy()
print(f' checkpoint["02_drop_nulls"] guardado — {checkpoints["02_drop_nulls"].shape}')
print('drop_high_null_cols OK ')

Columnas eliminadas (3): ['host_neighbourhood', 'neighbourhood_group_cleansed', 'calendar_updated']
 checkpoint["02_drop_nulls"] guardado — (9714, 76)
drop_high_null_cols OK 


---
## Task 3 — `clean_price`
Eliminar símbolo `$` y comas, convertir a `float`.  
Marcar como `NaN` precios iguales a 0.

In [8]:
df = checkpoints['02_drop_nulls'].copy()

df['price_num'] = (
    df['price']
    .str.replace(r'[\$,]', '', regex=True)
    .astype(float)
)

n_zero = (df['price_num'] == 0).sum()
df.loc[df['price_num'] == 0, 'price_num'] = np.nan
print(f'Precios igual a 0 anulados: {n_zero}')
print(df['price_num'].describe().round(2))

checkpoints['03_price'] = df.copy()
print(f' checkpoint["03_price"] guardado — {checkpoints["03_price"].shape}')
print('clean_price OK ')

Precios igual a 0 anulados: 0
count     8815.00
mean       285.79
std       1468.92
min         16.00
25%         76.00
50%        102.00
75%        147.00
max      92150.00
Name: price_num, dtype: float64
 checkpoint["03_price"] guardado — (9714, 77)
clean_price OK 


---
## Task 4 — `clean_booleans`
Homogeneizar columnas con valores `t`/`f` a `True`/`False`.

In [9]:
df = checkpoints['03_price'].copy()

BOOL_COLS = [
    'host_is_superhost',
    'host_has_profile_pic',
    'host_identity_verified',
    'instant_bookable',
    'has_availability'
]
bool_cols_present = [c for c in BOOL_COLS if c in df.columns]

for col in bool_cols_present:
    original_nulls = df[col].isnull().sum()
    df[col] = df[col].map({'t': True, 'f': False})
    new_nulls = df[col].isnull().sum()
    if new_nulls > original_nulls:
        print(f'  {col}: {new_nulls - original_nulls} valores inesperados → NaN')
    else:
        print(f'{col}: OK')

checkpoints['04_booleans'] = df.copy()
print(f'checkpoint["04_booleans"] guardado — {checkpoints["04_booleans"].shape}')
print('clean_booleans OK ')

host_is_superhost: OK
host_has_profile_pic: OK
host_identity_verified: OK
instant_bookable: OK
has_availability: OK
checkpoint["04_booleans"] guardado — (9714, 77)
clean_booleans OK 


---
## Task 5 — `clean_rates`
Eliminar `%` de `host_response_rate` y `host_acceptance_rate`, convertir a `float` en rango [0, 100].

In [10]:
df = checkpoints['04_booleans'].copy()

RATE_COLS = ['host_response_rate', 'host_acceptance_rate']

for col in RATE_COLS:
    if col not in df.columns:
        print(f'  {col} no encontrada, se omite')
        continue

    df[col + '_num'] = (
        df[col]
        .str.replace('%', '', regex=False)
        .astype(float)
    )

    out_of_range = (
        ~df[col + '_num'].between(0, 100, inclusive='both') &
        df[col + '_num'].notna()
    ).sum()
    if out_of_range:
        print(f'  {col}_num: {out_of_range} valores fuera de [0,100] → NaN')
        df.loc[~df[col + '_num'].between(0, 100, inclusive='both'), col + '_num'] = np.nan
    else:
        print(f'{col}_num: rango OK')

checkpoints['05_rates'] = df.copy()
print(f' checkpoint["05_rates"] guardado — {checkpoints["05_rates"].shape}')
print('clean_rates OK ')

host_response_rate_num: rango OK
host_acceptance_rate_num: rango OK
 checkpoint["05_rates"] guardado — (9714, 79)
clean_rates OK 


---
## Task 6 — `clean_dates`
Parsear columnas de fecha y detectar valores inválidos o inconsistentes.

In [ ]:


DATE_COLS = ['host_since', 'first_review', 'last_review'] 
# --- ESTADO ANTES (05_rates) ---
print("=== ESTADO ANTES (05_rates) ===")
print(f"Tipos iniciales:\n{checkpoints['05_rates'][DATE_COLS].dtypes}\n")
print(f"Nulos iniciales:\n{checkpoints['05_rates'][DATE_COLS].isnull().sum()}\n")

# --- PROCESO DE LIMPIEZA ---
df = checkpoints['05_rates'].copy()
DATE_COLS = ['host_since', 'first_review', 'last_review']

for col in DATE_COLS:
    if col not in df.columns:
        print(f'  {col} no encontrada, se omite')
        continue

    # Convertir a datetime
    df[col] = pd.to_datetime(df[col], errors='coerce')

    # Fechas futuras respecto al scraping → inválidas
    # (Asegúrate de que SCRAPE_DATE esté definido previamente como datetime)
    invalid_future = (df[col] > SCRAPE_DATE).sum()
    if invalid_future:
        print(f'  {col}: {invalid_future} fechas futuras detectadas → NaT')
        df.loc[df[col] > SCRAPE_DATE, col] = pd.NaT

# Inconsistencia lógica: first_review posterior a last_review
if 'first_review' in df.columns and 'last_review' in df.columns:
    invalid_order = (
        df['first_review'].notna() &
        df['last_review'].notna() &
        (df['first_review'] > df['last_review'])
    ).sum()
    
    if invalid_order:
        print(f'  Inconsistencia: first_review > last_review en {invalid_order} filas → Reseteando a NaT')
        mask = df['first_review'] > df['last_review']
        df.loc[mask, ['first_review', 'last_review']] = pd.NaT
    else:
        print('  Orden first_review / last_review: OK')

# Guardar resultado
checkpoints['06_dates'] = df.copy()

# --- ESTADO DESPUÉS (06_dates) ---
print("\n=== ESTADO DESPUÉS (06_dates) ===")
print(f"Tipos finales:\n{checkpoints['06_dates'][DATE_COLS].dtypes}\n")
print(f"Nulos finales (incluye coerciones e inválidos):\n{checkpoints['06_dates'][DATE_COLS].isnull().sum()}")
print(f"Dimensiones finales: {checkpoints['06_dates'].shape}")
print('clean_dates OK')

=== ESTADO ANTES (05_rates) ===
Tipos iniciales:
host_since      str
first_review    str
last_review     str
dtype: object

Nulos iniciales:
host_since         0
first_review    1005
last_review     1005
dtype: int64

  Orden first_review / last_review: OK

=== ESTADO DESPUÉS (06_dates) ===
Tipos finales:
host_since      datetime64[us]
first_review    datetime64[us]
last_review     datetime64[us]
dtype: object

Nulos finales (incluye coerciones e inválidos):
host_since         0
first_review    1005
last_review     1005
dtype: int64
Dimensiones finales: (9714, 79)
clean_dates OK


---
## Task 7 — `clean_bathrooms`
Validar la columna `bathrooms` numérica y cruzar con `bathrooms_text`.

In [19]:
import pandas as pd
import numpy as np
import re

print("=== ESTADO ANTES (06_dates) ===")
print(f"Nulos iniciales en 'bathrooms': {checkpoints['06_dates']['bathrooms'].isnull().sum()}")
if 'bathrooms_text' in checkpoints['06_dates'].columns:
    print(f"Ejemplos de 'bathrooms_text':\n{checkpoints['06_dates']['bathrooms_text'].dropna().head(3).tolist()}\n")

# --- PROCESO DE LIMPIEZA ---
df = checkpoints['06_dates'].copy()

# 1. Conversión y Rango Lógico
df['bathrooms'] = pd.to_numeric(df['bathrooms'], errors='coerce')

# Detectar valores fuera de rango [0, 20]
mask_invalid = ~df['bathrooms'].between(0, 20, inclusive='both') & df['bathrooms'].notna()
invalid_bath = mask_invalid.sum()

if invalid_bath:
    print(f"  -> {invalid_bath} valores fuera de [0,20] detectados. Seteando a NaN...")
    df.loc[mask_invalid, 'bathrooms'] = np.nan

# 2. Validación cruzada con bathrooms_text
if 'bathrooms_text' in df.columns:
    def parse_bathrooms_text(text):
        if pd.isna(text): return np.nan
        text = text.lower().strip()
        if 'half' in text or 'medio' in text: # Añadido 'medio' por si acaso
            return 0.5
        match = re.search(r'[\d\.]+', text)
        return float(match.group()) if match else np.nan

    # Creamos columna temporal para comparar
    df['bathrooms_text_num'] = df['bathrooms_text'].apply(parse_bathrooms_text)
    
    # Calcular discrepancias
    mask_mismatch = (
        df['bathrooms'].notna() & 
        df['bathrooms_text_num'].notna() & 
        (df['bathrooms'] != df['bathrooms_text_num'])
    )
    mismatch = mask_mismatch.sum()
    
    print(f"  -> Discrepancias encontradas: {mismatch}")
    
    # OPCIONAL: Si 'bathrooms' es NaN pero tenemos el texto, podemos recuperar el dato
    recovered = (df['bathrooms'].isna() & df['bathrooms_text_num'].notna()).sum()
    if recovered:
        print(f"  -> Recuperando {recovered} valores de 'bathrooms' desde el texto...")
        df['bathrooms'] = df['bathrooms'].fillna(df['bathrooms_text_num'])

# Guardar resultado
checkpoints['07_bathrooms'] = df.copy()

print("\n=== ESTADO DESPUÉS (07_bathrooms) ===")
print(f"Nulos finales en 'bathrooms': {df['bathrooms'].isnull().sum()}")
print(f"Checkpoint guardado: {checkpoints['07_bathrooms'].shape}")
print('clean_bathrooms OK')

=== ESTADO ANTES (06_dates) ===
Nulos iniciales en 'bathrooms': 899
Ejemplos de 'bathrooms_text':
['1 bath', '2 shared baths', '1.5 shared baths']

  -> Discrepancias encontradas: 1
  -> Recuperando 894 valores de 'bathrooms' desde el texto...

=== ESTADO DESPUÉS (07_bathrooms) ===
Nulos finales en 'bathrooms': 5
Checkpoint guardado: (9714, 80)
clean_bathrooms OK


---
## Task 8 — `engineer_features`
Crear variables derivadas necesarias para el pipeline.

In [13]:
df = checkpoints['07_bathrooms'].copy()

# Antigüedad del host en días respecto al scraping
df['host_tenure_days'] = (SCRAPE_DATE - df['host_since']).dt.days

# reviews_per_month → imputar a 0 si no tiene ninguna reseña (lógicamente correcto)
if 'reviews_per_month' in df.columns:
    no_reviews = df['number_of_reviews'] == 0
    df.loc[no_reviews, 'reviews_per_month'] = df.loc[no_reviews, 'reviews_per_month'].fillna(0)
    print(f'reviews_per_month imputados a 0: {no_reviews.sum()}')

print(f'host_tenure_days — nulos: {df["host_tenure_days"].isnull().sum()}')
print(f'host_tenure_days — rango: {df["host_tenure_days"].min()} – {df["host_tenure_days"].max()} días')

checkpoints['08_features'] = df.copy()
print(f' checkpoint["08_features"] guardado — {checkpoints["08_features"].shape}')
print('engineer_features OK ')

reviews_per_month imputados a 0: 1005
host_tenure_days — nulos: 0
host_tenure_days — rango: 8 – 5831 días
 checkpoint["08_features"] guardado — (9714, 81)
engineer_features OK 


---
## Task 9 — `flag_outliers`
Marcar outliers extremos con columnas flag.  
**No se eliminan filas** — la decisión queda para tasks posteriores del DAG.

In [14]:
df = checkpoints['08_features'].copy()

# Precio: outlier si supera el percentil 99.5
price_cap = df['price_num'].quantile(0.995)
df['price_outlier'] = df['price_num'] > price_cap
print(f'Outliers en precio (>{price_cap:.0f} €): {df["price_outlier"].sum()}')

# minimum_nights: outlier si supera 365
df['min_nights_outlier'] = df['minimum_nights'] > 365
print(f'Outliers en minimum_nights (>365): {df["min_nights_outlier"].sum()}')

# accommodates: outlier si > 16
df['accommodates_outlier'] = df['accommodates'] > 16
print(f'Outliers en accommodates (>16): {df["accommodates_outlier"].sum()}')

checkpoints['09_outliers'] = df.copy()
print(f' checkpoint["09_outliers"] guardado — {checkpoints["09_outliers"].shape}')
print('flag_outliers OK ')

Outliers en precio (>9429 €): 42
Outliers en minimum_nights (>365): 2
Outliers en accommodates (>16): 0
 checkpoint["09_outliers"] guardado — (9714, 84)
flag_outliers OK 


---
## Resumen final de checkpoints

In [15]:
print('=' * 55)
print(f'{"Checkpoint":<25} {"Filas":>8} {"Columnas":>10}')
print('=' * 55)
for name, ck in checkpoints.items():
    print(f'{name:<25} {ck.shape[0]:>8,} {ck.shape[1]:>10}')
print('=' * 55)

# Resumen de nulos en columnas clave del df final
df_final = checkpoints['09_outliers']
key_cols = [
    'price_num', 'accommodates', 'bedrooms', 'beds', 'bathrooms',
    'host_tenure_days', 'host_is_superhost',
    'host_response_rate_num', 'host_acceptance_rate_num',
    'review_scores_rating', 'neighbourhood_cleansed',
    'latitude', 'longitude'
]
key_cols = [c for c in key_cols if c in df_final.columns]

null_summary = df_final[key_cols].isnull().sum().rename('nulos').to_frame()
null_summary['%'] = (null_summary['nulos'] / len(df_final) * 100).round(1)
print('\nNulos en columnas clave (df final):')
print(null_summary)

print('\nPipeline de limpieza completado ')
print('df final disponible en: checkpoints["09_outliers"]')

Checkpoint                   Filas   Columnas
01_loaded                    9,714         79
02_drop_nulls                9,714         76
03_price                     9,714         77
04_booleans                  9,714         77
05_rates                     9,714         79
06_dates                     9,714         79
07_bathrooms                 9,714         80
08_features                  9,714         81
09_outliers                  9,714         84

Nulos en columnas clave (df final):
                          nulos     %
price_num                   899   9.3
accommodates                  0   0.0
bedrooms                    181   1.9
beds                        894   9.2
bathrooms                   899   9.3
host_tenure_days              0   0.0
host_is_superhost           265   2.7
host_response_rate_num      929   9.6
host_acceptance_rate_num    643   6.6
review_scores_rating       1005  10.3
neighbourhood_cleansed        0   0.0
latitude                      0   0.0
longitude

In [22]:
checkpoints['09_outliers'].head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,calculated_host_listings_count_shared_rooms,reviews_per_month,price_num,host_response_rate_num,host_acceptance_rate_num,bathrooms_text_num,host_tenure_days,price_outlier,min_nights_outlier,accommodates_outlier
0,96033,https://www.airbnb.com/rooms/96033,20250930030808,2025-09-30,city scrape,"Bonito piso a 200m de la playa, El Palo (Málaga)",Do you have a backpacker spirit and are lookin...,"200 metres from the beaches of El Palo, Malaga...",https://a0.muscache.com/pictures/hosting/Hosti...,510467,...,0,1.88,58.0,100.0,100.0,1.0,5282,False,False,False
1,166473,https://www.airbnb.com/rooms/166473,20250930030808,2025-09-30,city scrape,Perfect Location In Malaga,This apartment is rented out by the room - new...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,793360,...,0,0.59,28.0,100.0,88.0,2.0,5198,False,False,False
2,330760,https://www.airbnb.com/rooms/330760,20250930030808,2025-09-30,city scrape,Malaga Lodge Guesthouse Double room-shared bath.,The Lodge is set in a charming town house in L...,Málaga Lodge is situated next to the famous Sa...,https://a0.muscache.com/pictures/85419390/38a9...,1687526,...,0,0.41,60.0,91.0,100.0,1.5,4989,False,False,False
3,340024,https://www.airbnb.com/rooms/340024,20250930030808,2025-09-30,city scrape,NEW APARTMENT IN MALAGA CENTER,Welcome to Málaga!<br />This is a modern and e...,It is a central area and has all kinds of serv...,https://a0.muscache.com/pictures/hosting/Hosti...,1725690,...,0,2.11,61.0,100.0,100.0,1.0,4982,False,False,False
4,358541,https://www.airbnb.com/rooms/358541,20250930030808,2025-09-30,city scrape,Casa La Maga - Apartment for happy people,"For years, Raúl and I were super happy in this...",The apartment is in the very heart of Malaga C...,https://a0.muscache.com/pictures/miso/Hosting-...,1526932,...,0,2.48,87.0,86.0,78.0,1.0,5031,False,False,False
